In [1]:
#라이브러리 로드

import pandas as pd
import numpy as np
import json
import warnings
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import os
import ast

 
warnings.filterwarnings('ignore')
 
# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False



In [2]:
#데이터 로드 

# 데이터 경로
DATA_DIR = 'C:/project_3/Third-project-sparta/works/doha_Kim/raw_data'
TIERS = ['Platinum', 'Diamond', 'Master', 'GrandMaster', 'Challenger']

# 챔피언 정보 로드
champions_df = pd.read_csv(f'{DATA_DIR}/TFT_Champion_CurrentVersion.csv')
champions = champions_df.copy()
print(f"챔피언 정보: {len(champions_df)}")
 
# 아이템 정보 로드
items_df = pd.read_csv(f'{DATA_DIR}/TFT_Item_CurrentVersion.csv')
items = items_df.copy()
item_dict = dict(zip(items['id'], items['name']))
print(f"아이템 정보: {len(items_df)}")

# 각 티어별 데이터 로드
all_data = {}
for tier in TIERS:
    file_path = f'{DATA_DIR}/TFT_{tier}_MatchData.csv'
    try:
        df_raw = pd.read_csv(file_path) 
        df = df_raw.copy()
        df['tier'] = tier
        all_data[tier] = df
        print(f"{tier} 데이터: {len(df)}")
    except FileNotFoundError:
        print(f"{tier} 데이터 파일 없음")
 
# 전체 데이터 병합
full_data = pd.concat(all_data.values(), ignore_index=True)
print(f"전체 데이터: {len(full_data)} rows, {len(full_data.columns)}컬럼")

챔피언 정보: 52
아이템 정보: 54
Platinum 데이터: 80000
Diamond 데이터: 80000
Master 데이터: 79999
GrandMaster 데이터: 80000
Challenger 데이터: 79999
전체 데이터: 399998 rows, 9컬럼


In [3]:
full_data
# full_data.dtypes

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
0,KR_4291707834,1963.905273,6,27,5,1390.165771,"{'Cybernetic': 1, 'Demolitionist': 1, 'Infiltr...","{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {...",Platinum
1,KR_4291707834,1963.905273,8,37,3,1891.282715,"{'Blaster': 1, 'Chrono': 1, 'Cybernetic': 4, '...","{'Ziggs': {'items': [24], 'star': 3}, 'Fiora':...",Platinum
2,KR_4291707834,1963.905273,6,25,7,1279.461060,"{'Blaster': 1, 'Cybernetic': 1, 'DarkStar': 2,...","{'Fiora': {'items': [1], 'star': 1}, 'Shaco': ...",Platinum
3,KR_4291707834,1963.905273,7,38,2,1955.608521,"{'DarkStar': 1, 'Protector': 2, 'Set3_Blademas...","{'Poppy': {'items': [], 'star': 2}, 'Xayah': {...",Platinum
4,KR_4291707834,1963.905273,8,38,1,1955.608521,"{'Blaster': 1, 'Chrono': 5, 'DarkStar': 3, 'Pr...","{'TwistedFate': {'items': [36, 27], 'star': 3}...",Platinum
...,...,...,...,...,...,...,...,...,...
399993,KR_4357265434,2094.518555,7,33,4,1838.332764,"{'DarkStar': 2, 'Demolitionist': 2, 'Infiltrat...","{'KhaZix': {'items': [67, 26, 56], 'star': 2},...",Challenger
399994,KR_4357265434,2094.518555,8,33,5,1837.548096,"{'Blaster': 1, 'Chrono': 2, 'Cybernetic': 6, '...","{'Fiora': {'items': [35], 'star': 2}, 'Leona':...",Challenger
399995,KR_4357265434,2094.518555,9,33,6,1833.472046,"{'Chrono': 2, 'Cybernetic': 1, 'Demolitionist'...","{'TwistedFate': {'items': [], 'star': 2}, 'Mal...",Challenger
399996,KR_4357265434,2094.518555,8,31,7,1730.829712,"{'Blaster': 4, 'Chrono': 2, 'Cybernetic': 1, '...","{'Malphite': {'items': [], 'star': 1}, 'Graves...",Challenger


In [4]:
# 데이터 기본 정보
 
print("컬럼 정보:")
print(full_data.columns.tolist())
print('')
# print("데이터 샘플")
# print(full_data[['gameId', 'tier', 'Ranked', 'level', 'lastRound']].head(2))
print('')
print("결측치 확인:")
print(full_data.isnull().sum())
print('')
print("티어별 rows:")
print(full_data['tier'].value_counts().sort_index())

컬럼 정보:
['gameId', 'gameDuration', 'level', 'lastRound', 'Ranked', 'ingameDuration', 'combination', 'champion', 'tier']


결측치 확인:
gameId            0
gameDuration      0
level             0
lastRound         0
Ranked            0
ingameDuration    0
combination       0
champion          0
tier              0
dtype: int64

티어별 rows:
tier
Challenger     79999
Diamond        80000
GrandMaster    80000
Master         79999
Platinum       80000
Name: count, dtype: int64


In [5]:
game_counts = full_data.groupby('gameId').size()
print(game_counts)

gameId
KR_3890408252    8
KR_3891371111    8
KR_3891442705    8
KR_3891808329    8
KR_3965402291    8
                ..
KR_4402761495    8
KR_4402771106    8
KR_4402800301    8
KR_4402801838    8
KR_4402801990    8
Length: 49981, dtype: int64


In [6]:
# 중복행
duplicates = full_data[full_data.duplicated(keep=False)]

print(f"중복 제거 전: {len(full_data)}")
print(f"중복된 행 개수: {len(duplicates)}")
print()

# 중복행 샘플
print(duplicates.head(20))
print()

# 중복 제거
full_data = full_data.drop_duplicates()
print(f"중복 제거 후: {len(full_data)}")

중복 제거 전: 399998
중복된 행 개수: 80

              gameId  gameDuration  level  lastRound  Ranked  ingameDuration  \
657    KR_4229542485    127.697548      3          4       0      127.697548   
658    KR_4229542485    127.697548      3          4       0      127.697548   
659    KR_4229542485    127.697548      3          4       0      127.697548   
660    KR_4229542485    127.697548      3          4       0      127.697548   
661    KR_4229542485    127.697548      3          4       0      127.697548   
662    KR_4229542485    127.697548      3          4       0      127.697548   
663    KR_4229542485    127.697548      3          4       0      127.697548   
18009  KR_4236285365    452.641998      4         10       0      452.641998   
18010  KR_4236285365    452.641998      4         10       0      452.641998   
18011  KR_4236285365    452.641998      4         10       0      452.641998   
18012  KR_4236285365    452.641998      4         10       0      452.641998   
18013  KR_

In [7]:
game_counts = full_data.groupby('gameId').size()
print(game_counts)

gameId
KR_3890408252    1
KR_3891371111    1
KR_3891442705    1
KR_3891808329    1
KR_3965402291    8
                ..
KR_4402761495    8
KR_4402771106    8
KR_4402800301    8
KR_4402801838    8
KR_4402801990    8
Length: 49981, dtype: int64


In [8]:
invalid_games = game_counts[game_counts != 8].reset_index()
invalid_games.columns = ['gameId', 'count']
# # invalid_games.count() # 33개의 게임id

invalid_game_ids = invalid_games['gameId'].tolist()
tier_list = full_data[full_data['gameId'].isin(invalid_game_ids)][['gameId', 'tier']]

# gameId별 tier별 인원수 계산
tier_list = tier_list.groupby(['gameId', 'tier']).size().reset_index(name='tier_count')

# merge 
invalid_games = invalid_games.merge(tier_list, on='gameId')
print(invalid_games)


           gameId  count         tier  tier_count
0   KR_3890408252      1     Platinum           1
1   KR_3891371111      1     Platinum           1
2   KR_3891442705      1     Platinum           1
3   KR_3891808329      1     Platinum           1
4   KR_4229542485      2     Platinum           2
5   KR_4231144753      6     Platinum           6
6   KR_4231208133      3     Platinum           3
7   KR_4236285365      3     Platinum           3
8   KR_4263820118     16       Master           8
9   KR_4263820118     16     Platinum           8
10  KR_4295022760      3       Master           3
11  KR_4303195386      3       Master           3
12  KR_4313697214     16       Master           8
13  KR_4313697214     16     Platinum           8
14  KR_4318806255      7   Challenger           7
15  KR_4320079059     16      Diamond           8
16  KR_4320079059     16     Platinum           8
17  KR_4323594767      2     Platinum           2
18  KR_4335870255      7       Master           7


In [9]:
filtered_df = full_data[full_data["gameId"].isin(invalid_game_ids)]
filtered_df = filtered_df.sort_values(by="gameId")

# pd.set_option('display.max_rows', None)
filtered_df

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
23616,KR_3890408252,0.000000,1,0,0,0.000000,{},{},Platinum
62192,KR_3891371111,0.000000,1,0,0,0.000000,{},{},Platinum
62184,KR_3891442705,0.000000,1,0,0,0.000000,{},{},Platinum
62176,KR_3891808329,0.000000,1,0,0,0.000000,{},{},Platinum
656,KR_4229542485,127.697548,3,4,8,125.573448,"{'Chrono': 1, 'Cybernetic': 1, 'Sniper': 1, 'V...","{'Caitlyn': {'items': [], 'star': 1}, 'Leona':...",Platinum
...,...,...,...,...,...,...,...,...,...
135899,KR_4398618214,2237.413330,8,30,7,1623.692383,"{'Blaster': 1, 'Chrono': 1, 'DarkStar': 1, 'De...","{'Jayce': {'items': [], 'star': 2}, 'Kassadin'...",Diamond
135900,KR_4398618214,2237.413330,8,33,5,1799.782227,"{'Chrono': 1, 'Cybernetic': 1, 'Infiltrator': ...","{'Malphite': {'items': [], 'star': 2}, 'KhaZix...",Diamond
135901,KR_4398618214,2237.413330,8,40,3,2177.933350,"{'Chrono': 2, 'Cybernetic': 1, 'DarkStar': 1, ...","{'JarvanIV': {'items': [67, 27, 36], 'star': 3...",Diamond
6335,KR_4398618214,2237.413330,8,35,4,1927.595215,"{'Blaster': 2, 'Demolitionist': 3, 'Mercenary'...","{'Ziggs': {'items': [44, 44, 33], 'star': 3}, ...",Platinum


In [10]:
# gameDuration = 0
zero_duration_ids = set(full_data[full_data['gameDuration'] == 0]['gameId'])
# zero_duration_ids #{'KR_3890408252', 'KR_3891371111', 'KR_3891442705', 'KR_3891808329'}
# ddf = full_data[full_data["gameId"].isin(zero_duration_ids)]
# ddf

# gameId	gameDuration	level	lastRound	Ranked	ingameDuration	combination	champion	tier
# 23616	KR_3890408252	0.0	1	0	0	0.0	{}	{}	Platinum
# 62176	KR_3891808329	0.0	1	0	0	0.0	{}	{}	Platinum
# 62184	KR_3891442705	0.0	1	0	0	0.0	{}	{}	Platinum
# 62192	KR_3891371111	0.0	1	0	0	0.0	{}	{}	Platinum

# combination = '{}'
def parse_combination(combo_str):
    """조합 문자열을 딕셔너리로 변환"""
    try:
        if pd.isna(combo_str):
            return {}
        if isinstance(combo_str, str) and combo_str.startswith('{'):
            return ast.literal_eval(combo_str)
        return {}
    except:
        return {}
    
# 적용
full_data['combination_dict'] = full_data['combination'].apply(parse_combination)

empty_combination_ids = set(full_data[full_data['combination_dict'] == {}]['gameId'])
empty_combination_ids
len(empty_combination_ids) ## gameDuration = 0
zero_duration_ids = set(full_data[full_data['gameDuration'] == 0]['gameId'])
# zero_duration_ids #{'KR_3890408252', 'KR_3891371111', 'KR_3891442705', 'KR_3891808329'}
# ddf = full_data[full_data["gameId"].isin(zero_duration_ids)]
# ddf

# gameId	gameDuration	level	lastRound	Ranked	ingameDuration	combination	champion	tier
# 23616	KR_3890408252	0.0	1	0	0	0.0	{}	{}	Platinum
# 62176	KR_3891808329	0.0	1	0	0	0.0	{}	{}	Platinum
# 62184	KR_3891442705	0.0	1	0	0	0.0	{}	{}	Platinum
# 62192	KR_3891371111	0.0	1	0	0	0.0	{}	{}	Platinum

# combination = '{}'
def parse_combination(combo_str):
    """조합 문자열을 딕셔너리로 변환"""
    try:
        if pd.isna(combo_str):
            return {}
        if isinstance(combo_str, str) and combo_str.startswith('{'):
            return ast.literal_eval(combo_str)
        return {}
    except:
        return {}
    
# 적용
full_data['combination_dict'] = full_data['combination'].apply(parse_combination)

empty_combination_ids = set(full_data[full_data['combination_dict'] == {}]['gameId'])
empty_combination_ids
len(empty_combination_ids) #313



313

In [11]:
# champion = '{}'
# 적용
full_data['champion_dict'] = full_data['champion'].apply(parse_combination)
empty_champion_ids = set(full_data[full_data['champion_dict'] == {}]['gameId'])
empty_champion_ids
len(empty_champion_ids) #240


240

In [12]:
# 결측 갯수
all_issues_ids = set(full_data[
    (full_data['gameDuration'] == 0) | 
    (full_data['combination_dict'] == {}) | 
    (full_data['champion_dict'] == {})
]['gameId'])
all_issues_ids
len(all_issues_ids) #347

347

In [13]:
full_data.dtypes

gameId                  str
gameDuration        float64
level                 int64
lastRound             int64
Ranked                int64
ingameDuration      float64
combination             str
champion                str
tier                    str
combination_dict     object
champion_dict        object
dtype: object

In [14]:
# Combination 테이블 생성
combination_data = []
for idx, row in full_data.iterrows():
    try:
        combo_dict = ast.literal_eval(row['combination'])
    except:
        combo_dict = {}
    
    for trait_name, trait_count in combo_dict.items():
        combination_data.append({
            'gameId': row['gameId'],
            'tier': row['tier'],
            'Ranked': row['Ranked'],
            'trait_name': trait_name,
            'trait_count': trait_count
        })

combination_table = pd.DataFrame(combination_data)

In [15]:
combination_table

,gameId,tier,Ranked,trait_name,trait_count
0,KR_4291707834,Platinum,5,Cybernetic,1
1,KR_4291707834,Platinum,5,Demolitionist,1
2,KR_4291707834,Platinum,5,Infiltrator,1
3,KR_4291707834,Platinum,5,Rebel,1
4,KR_4291707834,Platinum,5,Set3_Brawler,1
...,...,...,...,...,...
3456602,KR_4357265434,Challenger,8,Protector,2
3456603,KR_4357265434,Challenger,8,Set3_Celestial,4
3456604,KR_4357265434,Challenger,8,Sniper,1
3456605,KR_4357265434,Challenger,8,SpacePirate,2


In [17]:
combination_table_sorted = combination_table.sort_values(by="trait_name")
combination_table_sorted 

,gameId,tier,Ranked,trait_name,trait_count
519518,KR_3997876981,Platinum,4,Alchemist,1
510995,KR_4206177777,Platinum,3,Alchemist,1
265448,KR_4194991391,Platinum,6,Alchemist,1
1158130,KR_4115440002,Diamond,4,Alchemist,1
7299,KR_4217035798,Platinum,2,Alchemist,1
...,...,...,...,...,...
467463,KR_4067871701,Platinum,7,Woodland,1
389979,KR_4120089014,Platinum,2,Woodland,3
172281,KR_4131723850,Platinum,8,Woodland,3
1358486,KR_4118691766,Diamond,3,Woodland,4


In [ ]:
#Champion 테이블 생성
champion_data = []

for idx, row in full_data.iterrows():
    game_id = row['gameId']
    tier = row['tier']
    ranked = row['Ranked']
    
    # 문자열 딕셔너리 변환
    try:
        champion_dict = ast.literal_eval(row['champion'])
    except:
        champion_dict = {}
    
    # 각 챔피언별 행 생성
    for champion_name, champion_info in champion_dict.items():
        items = champion_info.get('items', [])  # 아이템 리스트
        star = champion_info.get('star', 0)     # 챔피언 레벨
        
        champion_data.append({
            'gameId': game_id,
            'tier': tier,
            'Ranked': ranked,
            'champion_name': champion_name,
            'items': items,      # 리스트로 유지
            'star': star
        })

# 데이터프레임 생성
champion_table = pd.DataFrame(champion_data)

In [19]:
champion_table

,gameId,tier,Ranked,champion_name,items,star
0,KR_4291707834,Platinum,5,Ziggs,[7],1
1,KR_4291707834,Platinum,5,Ashe,[9],1
2,KR_4291707834,Platinum,5,ChoGath,[6],1
3,KR_4291707834,Platinum,5,Ekko,[1],1
4,KR_4291707834,Platinum,3,Ziggs,[24],3
...,...,...,...,...,...,...
3156810,KR_4357265434,Challenger,8,Rakan,[69],3
3156811,KR_4357265434,Challenger,8,XinZhao,[1],3
3156812,KR_4357265434,Challenger,8,Jayce,"[37, 15, 39]",2
3156813,KR_4357265434,Challenger,8,Kassadin,[26],2


In [ ]:
#Champion 테이블에 Item 정보 추가
champion_table_with_items = champion_table.copy()

# 아이템 개수 계산
champion_table_with_items['item_count'] = champion_table_with_items['items'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

# explode
champion_table_exploded = champion_table_with_items.explode('items', ignore_index=True)

# 숫자 변환 (0 포함)
champion_table_exploded['item_id'] = pd.to_numeric(
    champion_table_exploded['items'], errors='coerce'
).astype('Int64')

# 매핑
champion_table_exploded['item_name'] = champion_table_exploded['item_id'].map(item_dict)

# 정리
champion_table_exploded = champion_table_exploded[[
    'gameId', 'tier', 'Ranked', 'champion_name', 'item_id', 'item_name', 'star', 'item_count'
]]

In [23]:
champion_table_exploded

,gameId,tier,Ranked,champion_name,item_id,item_name,star,item_count
0,KR_4291707834,Platinum,5,Ziggs,7,Giant's Belt,1,1
1,KR_4291707834,Platinum,5,Ashe,9,Sparring Gloves,1,1
2,KR_4291707834,Platinum,5,ChoGath,6,Negatron Cloak,1,1
3,KR_4291707834,Platinum,5,Ekko,1,B.F. Sword,1,1
4,KR_4291707834,Platinum,3,Ziggs,24,Statikk Shiv,3,1
...,...,...,...,...,...,...,...,...
4902246,KR_4357265434,Challenger,8,Jayce,37,Morellonomicon,2,3
4902247,KR_4357265434,Challenger,8,Jayce,15,Guardian Angel,2,3
4902248,KR_4357265434,Challenger,8,Jayce,39,Jeweled Gauntlet,2,3
4902249,KR_4357265434,Challenger,8,Kassadin,26,Runaan's Hurricane,2,1


In [25]:
champion_table_exploded[champion_table_exploded['item_id'] == 0]
# (champion_table_exploded['item_id'] == 0).sum()

,gameId,tier,Ranked,champion_name,item_id,item_name,star,item_count


In [ ]:
#Champion 테이블에 Champions 파일 정보 추가
#키 값 통일
champion_table['champion_name'] = champion_table['champion_name'].str.strip().str.lower()
champions['name'] = champions['name'].str.strip().str.lower()

# champion_table과 champions 테이블 merge
champion_with_details = champion_table.merge(
    champions[['name', 'cost', 'speed_of_attack', 'skill_name', 'skill_cost', 'origin', 'class']],
    left_on='champion_name',
    right_on='name',
    how='left'  
)

# 중복 'name' 칼럼 제거
# champion_with_details = champion_with_details.drop(columns=['name'])

# 칼럼 순서 정렬
champion_with_details = champion_with_details[[
    'gameId', 'tier', 'Ranked', 'champion_name', 'items', 'star',
    'cost', 'speed_of_attack', 'skill_name', 'skill_cost', 'origin', 'class'
]]

In [33]:
champion_with_details.head(50)

,gameId,tier,Ranked,champion_name,items,star,cost,speed_of_attack,skill_name,skill_cost,origin,class
0,KR_4291707834,Platinum,5,ziggs,[7],1,1.0,0.70,ziggs_bomb,0/45,Rebel,['Demolitionist']
1,KR_4291707834,Platinum,5,ashe,[9],1,3.0,0.80,ashe_enchantedcrystalarrow,50/125,Celestial,['Sniper']
2,KR_4291707834,Platinum,5,chogath,[6],1,4.0,0.60,chogath_rupture,50/150,Void,['Brawler']
3,KR_4291707834,Platinum,5,ekko,[1],1,5.0,0.90,ekko_chronobreak,50/150,Cybernetic,['Infiltrato']
4,KR_4291707834,Platinum,3,ziggs,[24],3,1.0,0.70,ziggs_bomb,0/45,Rebel,['Demolitionist']
5,KR_4291707834,Platinum,3,fiora,[37],2,1.0,1.00,fiora_reposte,0/85,Cybernetic,['Blademaster']
6,KR_4291707834,Platinum,3,leona,"[36, 24]",2,1.0,0.55,leona_cyberbarrier,50/100,Cybernetic,['Vanguard']
7,KR_4291707834,Platinum,3,lucian,[],2,2.0,0.70,lucian_relentlesspersuit,0/35,Cybernetic,['Blaster']
8,KR_4291707834,Platinum,3,vi,[5],2,3.0,0.65,vi_assaultandbattery,70/140,Cybernetic,['Brawler']
9,KR_4291707834,Platinum,3,kayle,[],2,4.0,0.80,kayle_ascend,0/60,Valkyrie,['Blademaster']


In [44]:
# Class 칼럼 처리
champion_with_class_joined = champion_with_details.copy()

# 리스트 문자열 변환
champion_with_class_joined['class'] = champion_with_class_joined['class'].apply(
    lambda x: ', '.join(ast.literal_eval(x)) if isinstance(x, str) else ', '.join(x if isinstance(x, list) else [])
)

In [45]:
champion_with_class_joined

,gameId,tier,Ranked,champion_name,items,star,cost,speed_of_attack,skill_name,skill_cost,origin,class
0,KR_4291707834,Platinum,5,ziggs,[7],1,1.0,0.70,ziggs_bomb,0/45,Rebel,Demolitionist
1,KR_4291707834,Platinum,5,ashe,[9],1,3.0,0.80,ashe_enchantedcrystalarrow,50/125,Celestial,Sniper
2,KR_4291707834,Platinum,5,chogath,[6],1,4.0,0.60,chogath_rupture,50/150,Void,Brawler
3,KR_4291707834,Platinum,5,ekko,[1],1,5.0,0.90,ekko_chronobreak,50/150,Cybernetic,Infiltrato
4,KR_4291707834,Platinum,3,ziggs,[24],3,1.0,0.70,ziggs_bomb,0/45,Rebel,Demolitionist
...,...,...,...,...,...,...,...,...,...,...,...,...
3156810,KR_4357265434,Challenger,8,rakan,[69],3,2.0,0.70,rakan_grandentrance,50/100,Celestial,Protector
3156811,KR_4357265434,Challenger,8,xinzhao,[1],3,2.0,0.70,xinzhao_threetalonstrike,0/60,Celestial,Protector
3156812,KR_4357265434,Challenger,8,jayce,"[37, 15, 39]",2,3.0,0.70,jayce_totheskies,0/80,Space Pirate,Vanguard
3156813,KR_4357265434,Challenger,8,kassadin,[26],2,3.0,0.65,kassadin_forcepulse,40/80,Celestial,Mana-Reaver


In [ ]:
# multi_class_champions = champion_with_class_joined[
#     champion_with_class_joined['class'].str.contains(',', na=False)
# ]
# multi_class_champions.head()

,gameId,tier,Ranked,champion_name,items,star,cost,speed_of_attack,skill_name,skill_cost,origin,class
15,KR_4291707834,Platinum,7,missfortune,[3],1,5.0,1.00,missfortune_bullettime,75/175,Valkyrie,"Mercenary, Blaster"
62,KR_4291614366,Platinum,5,irelia,[5],2,4.0,0.85,irelia_bladesurge,0/30,Cybernetic,"Mana-Reaver, Blademaster"
64,KR_4291614366,Platinum,5,missfortune,[27],1,5.0,1.00,missfortune_bullettime,75/175,Valkyrie,"Mercenary, Blaster"
71,KR_4291614366,Platinum,6,irelia,[],1,4.0,0.85,irelia_bladesurge,0/30,Cybernetic,"Mana-Reaver, Blademaster"
84,KR_4291614366,Platinum,1,missfortune,[56],2,5.0,1.00,missfortune_bullettime,75/175,Valkyrie,"Mercenary, Blaster"


In [ ]:
# # Class 칼럼 처리 #원핫인코딩 버전
# champion_for_onehot = champion_with_details.copy()

# # 문자열 리스트 변환
# champion_for_onehot['class_list'] = champion_for_onehot['class'].apply(
#     lambda x: ast.literal_eval(x) if isinstance(x, str) else (x if isinstance(x, list) else [])
# )

# all_classes = set()
# for class_list in champion_for_onehot['class_list']:
#     if isinstance(class_list, list):
#         all_classes.update(class_list)

# all_classes = sorted(list(all_classes))
# # 모든 직업
# # ['Blademaster', 'Blaster', 'Brawler', 'Demolitionist', 'Infiltrato', 'Infiltrator', 'Mana-Reaver', 'Mercenary', 'Mystic', 'Protector', 'Sniper', 'Sorcerer', 'Starship', 'Vanguard']

# # One-Hot Encoding 생성
# for class_name in all_classes:
#     champion_for_onehot[f'class_{class_name}'] = champion_for_onehot['class_list'].apply(
#         lambda x: 1 if isinstance(x, list) and class_name in x else 0
#     )

# # 원본 class, class_list 칼럼 제거
# champion_for_onehot = champion_for_onehot.drop(columns=['class', 'class_list'])
# champion_for_onehot 

,gameId,tier,Ranked,champion_name,items,star,cost,speed_of_attack,skill_name,skill_cost,...,class_Infiltrato,class_Infiltrator,class_Mana-Reaver,class_Mercenary,class_Mystic,class_Protector,class_Sniper,class_Sorcerer,class_Starship,class_Vanguard
0,KR_4291707834,Platinum,5,ziggs,[7],1,1.0,0.70,ziggs_bomb,0/45,...,0,0,0,0,0,0,0,0,0,0
1,KR_4291707834,Platinum,5,ashe,[9],1,3.0,0.80,ashe_enchantedcrystalarrow,50/125,...,0,0,0,0,0,0,1,0,0,0
2,KR_4291707834,Platinum,5,chogath,[6],1,4.0,0.60,chogath_rupture,50/150,...,0,0,0,0,0,0,0,0,0,0
3,KR_4291707834,Platinum,5,ekko,[1],1,5.0,0.90,ekko_chronobreak,50/150,...,1,0,0,0,0,0,0,0,0,0
4,KR_4291707834,Platinum,3,ziggs,[24],3,1.0,0.70,ziggs_bomb,0/45,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3156810,KR_4357265434,Challenger,8,rakan,[69],3,2.0,0.70,rakan_grandentrance,50/100,...,0,0,0,0,0,1,0,0,0,0
3156811,KR_4357265434,Challenger,8,xinzhao,[1],3,2.0,0.70,xinzhao_threetalonstrike,0/60,...,0,0,0,0,0,1,0,0,0,0
3156812,KR_4357265434,Challenger,8,jayce,"[37, 15, 39]",2,3.0,0.70,jayce_totheskies,0/80,...,0,0,0,0,0,0,0,0,0,1
3156813,KR_4357265434,Challenger,8,kassadin,[26],2,3.0,0.65,kassadin_forcepulse,40/80,...,0,0,1,0,0,0,0,0,0,0


In [ ]:
#전처리 정리후 보면 좋을 것들 

In [ ]:
# # 특정 조합의 출현 빈도
# print(combination_table['trait_name'].value_counts())

# # Platinum 티어에서 가장 많이 사용된 조합
# plat_combos = combination_table[combination_table['tier'] == 'Platinum']
# print(plat_combos.groupby('trait_name')['trait_count'].sum().sort_values(ascending=False))

# # 특정 게임의 모든 조합
# print(combination_table[combination_table['gameId'] == 'KR_4291707834'])

In [ ]:
# # 가장 많이 등장한 챔피언 TOP 10
# print(champion_table['champion_name'].value_counts().head(10))

# # 3성 챔피언의 개수
# three_star = champion_table[champion_table['star'] == 3]
# print(f"3성 챔피언 개수: {len(three_star)}")

# # 특정 게임의 모든 챔피언
# print(champion_table[champion_table['gameId'] == 'KR_4291707834'])

In [ ]:
# # 가장 많이 사용된 아이템
# print(champion_table_exploded['item_name'].value_counts().head(10))

# # 특정 챔피언이 가진 아이템들
# ziggs_items = champion_table_exploded[champion_table_exploded['champion_name'] == 'Ziggs']
# print(ziggs_items['item_name'].value_counts())

# # 각 아이템의 평균 챔피언 레벨(star)
# print(champion_table_exploded.groupby('item_name')['star'].mean().sort_values(ascending=False))

In [ ]:
# # 비용별 평균 star (레벨)
# print(champion_with_details.groupby('cost')['star'].mean())

# # 각 출신(origin)별 출현 빈도
# print(champion_with_details['origin'].value_counts())

# # Platinum 티어에서 비용이 5인 챔피언들
# print(champion_with_details[
#     (champion_with_details['tier'] == 'Platinum') & 
#     (champion_with_details['cost'] == 5)
# ])

In [ ]:
# # 특정 조합의 직업을 찾기
# mercenary_blaster = champion_with_class_joined[
#     champion_with_class_joined['class'].str.contains('Mercenary.*Blaster')
# ]

# # 직업 조합 빈도
# print(champion_with_class_joined['class'].value_counts())